In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
# Datasets and Dataloaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image -> scale (0, 1) -> normalize -> (-1, 1)
transfom = transforms.Compose([
    transforms.ToTensor(), # convert pytorch tensor + scale (0, 1)
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # standard deviation, mean value
])

train_set = CIFAR10(root="./data", train=True, download=True, transform=transfom)
test_set = CIFAR10(root="./data", train=False, download=True, transform=transfom)

100%|██████████| 170M/170M [00:59<00:00, 2.84MB/s] 


In [3]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

#### Building the CNN

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 1st Layer

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 2nd Layer

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 3rd Layer
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # Flattening step
        x = self.fc_layers(x)

        return x


In [5]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [6]:
# Training the CNN
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()

        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update parameters

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} and loss = {epoch_training_loss/len(train_loader)}")

epoch=1/10 and loss = 1.390390864266154
epoch=2/10 and loss = 0.9373434912746824
epoch=3/10 and loss = 0.7528314670867018
epoch=4/10 and loss = 0.618441270669098
epoch=5/10 and loss = 0.5096592070044154
epoch=6/10 and loss = 0.41185220344292234
epoch=7/10 and loss = 0.32375512777082144
epoch=8/10 and loss = 0.25215606863522316
epoch=9/10 and loss = 0.19552142179721152
epoch=10/10 and loss = 0.1496819647295815


In [7]:
# Evaluate CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels / total_labels * 100}")

Accuracy = 74.2
